# Visualize Traversals
This notebook contains some code to generate some figures to show the traversals on top of the greenspace data.

In [ ]:
import os
import json

import numpy as np
import rasterio as rio
import matplotlib.pyplot as plt
import geopandas as gpd

from matplotlib.colors import BoundaryNorm, ListedColormap

In [ ]:
def make_plot(green_pth, trav_pth, name):
    src = rio.open(green_pth)
    data = src.read(
        1,
        out_shape=(src.height // 10, src.width // 10),
        resampling=rio.enums.Resampling.nearest
    )
    gdf = gpd.read_file(trav_pth).to_crs(src.crs)

    transform = src.transform
    scale_x = src.width  / data.shape[1]
    scale_y = src.height / data.shape[0]

    left   = transform.c
    right  = transform.c + transform.a * src.width
    bottom = transform.f + transform.e * src.height
    top    = transform.f
    extent = [left, right, bottom, top]

    # --- Plot ---
    class_info = {
        0:  ('Not vegetated',                '#b2b2b2', 0.5),
        2:  ('Evergreen broadleaved forest', '#1a7a3c', 2.5),
        4:  ('Deciduous broadleaved forest', '#78c679', 4.5),
        6:  ('Evergreen needleleaved forest','#005a24', 6.5),
        8:  ('Deciduous needleleaved forest','#41ab5d', 8.5),
        10: ('Mixed-leaf forest',            '#addd8e', 10.5),
        12: ('Evergreen shrubland',          '#8c6d31', 12),
        13: ('Deciduous shrubland',          '#c9a96e', 13),
        14: ('Grassland',                    '#f7e08a', 14),
    }

    classes = list(class_info.keys())
    labels  = [v[0] for v in class_info.values()]
    colors  = [v[1] for v in class_info.values()]
    tickmarks = [v[2] for v in class_info.values()]

    cmap   = ListedColormap(colors)
    bounds = [c - 0.5 for c in classes] + [classes[-1] + 0.5]
    norm   = BoundaryNorm(bounds, cmap.N)

    fig, ax = plt.subplots(figsize=(10, 8))

    # Raster layer
    im = ax.imshow(data, cmap=cmap, norm=norm,
                interpolation='nearest', extent=extent)

    # Points layer
    gdf.plot(ax=ax,
            color='black',
            edgecolor='black',
            markersize=10,
            linewidth=0.5,
            zorder=5, label="Traversal")          # zorder ensures points render above raster

    cbar = fig.colorbar(im, ax=ax, ticks=tickmarks, shrink=0.8)
    cbar.ax.set_yticklabels(labels, fontsize=9)
    cbar.minorticks_off()

    ax.set_xlim(left, right)
    ax.set_ylim(bottom, top)
    ax.set_title(name)
    ax.axis('off')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join("../results/data_viz", f"{name}.pdf"), bbox_inches='tight')
    plt.close()

In [32]:
with open('../data/data_map.json') as src:
    dm = json.load(src)


In [ ]:
# Iterate over and make all of 
for e in dm['cities']:
    green_pth = os.path.join('..', e['greenspace'])
    trav_pth = os.path.join('..', e['traversal'], 'pm', 'trav.shp')
    name = e['name']
    make_plot(green_pth, trav_pth, name)

In [31]:
files_created = os.listdir('../results/data_viz')
print(f" There were {len(files_created)} files created.")

 There were 22 files created.
